<a href="https://colab.research.google.com/github/Bryan-625/DS340W-Project/blob/main/FinGPT_Inference_Llama2_13B_falcon_7B_for_Beginners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers==4.32.0 peft==0.5.0
!pip install sentencepiece
!pip install accelerate
!pip install torch
!pip install peft
!pip install datasets
!pip install bitsandbytes
!pip install -q -U torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 33.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.9 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 83.9 MB/s  0:00:00


In [ ]:
#for the `load_in_8bit=True` error
# !pip install protobuf transformers==4.32.0 cpm_kernels torch>=2.0 gradio mdtex2html sentencepiece accelerate

## Inference with Single Task: Sentiment (Original Implementation) (DO NOT RUN)

In [3]:
# from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast, BitsAndBytesConfig
# from peft import PeftModel  # 0.5.0

W1114 22:15:39.567000 6496 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1114 22:15:39.609000 6496 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
# Load Models
# base_model = "NousResearch/Llama-2-13b-hf"
# peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"
# tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# quant_config = BitsAndBytesConfig(load_in_8bit=True)
# model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0", quantization_config=quant_config,)
# model = PeftModel.from_pretrained(model, peft_model)
# model = model.eval()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 19.7MB            

adapter_model.bin: downloading bytes:           |  0.00B            

In [5]:
# Make prompts
# prompt = [
# '''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
# Input: FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is aggressively pursuing its growth strategy by increasingly focusing on technologically more demanding HDI printed circuit boards PCBs .
# Answer: ''',
# '''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
# Input: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
# Answer: ''',
# '''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
# Input: A tinyurl link takes users to a scamming site promising that users can earn thousands of dollars by becoming a Google ( NASDAQ : GOOG ) Cash advertiser .
# Answer: ''',
# ]

# Generate results
# tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
# res = model.generate(**tokens, max_length=512)
# res_sentences = [tokenizer.decode(i) for i in res]
# out_text = [o.split("Answer: ")[1] for o in res_sentences]

# show results
# for sentiment in out_text:
#     print(sentiment)

# Output:
# positive
# neutral
# negative


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2387: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


 positive</s>
 neutral</s>
 neutral</s>


## New Implementation

In [25]:
# Loading Libraries

import pandas as pd
import torch
import urllib.parse
import requests
import xml.etree.ElementTree as ET
import re
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaTokenizerFast, LlamaForCausalLM, BitsAndBytesConfig, LlamaTokenizerFast, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import PeftModel
from datasets import Dataset
from bs4 import BeautifulSoup

In [3]:
# Load FinGPT Model (Llama-2-13b + LoRA Adapter)

base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"

tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(load_in_8bit=True)

fingpt_model = LlamaForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    device_map="cuda:0",
    quantization_config=quant_config,
)
fingpt_model = PeftModel.from_pretrained(fingpt_model, peft_model)
fingpt_model = fingpt_model.eval()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 19.7MB            

adapter_model.bin: downloading bytes:           |  0.00B            

In [4]:
# Load Web Filter Model (Used to filter noise in retrieved documents)

# WebFilter classifies retrieved web documents as 'Reliable/Relevant' vs 'Unreliable/Irrelevant'
web_filter_model_name = "dayll/WebFilter-7B"
filter_tokenizer = AutoTokenizer.from_pretrained(web_filter_model_name, trust_remote_code=True)
filter_model = AutoModelForSequenceClassification.from_pretrained(
    web_filter_model_name,
    device_map="cuda:0"
)
filter_model = filter_model.eval()

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: dayll/WebFilter-7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# Training the Web Filter Model

# 1. Prepare training, testing, and validation data
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
val_df = pd.read_csv('/content/validation.csv')

# Handle missing or empty values
train_df = train_df.dropna(subset=['input', 'output'])
test_df = test_df.dropna(subset=['input', 'output'])
val_df = val_df.dropna(subset=['input', 'output'])

# Map 'input' to 'text'
train_df['text'] = train_df['input'].astype(str)
test_df['text'] = test_df['input'].astype(str)
val_df['text'] = val_df['input'].astype(str)

# Map your 3 sentiment classes ('positive', 'neutral', 'negative') to numerical IDs (0, 1, 2)
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['output'].map(label_mapping)
test_df['label'] = test_df['output'].map(label_mapping)
val_df['label'] = val_df['output'].map(label_mapping)

# Drop any rows that couldn't be mapped successfully
train_df = train_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])
val_df = val_df.dropna(subset=['label'])

train_df['label'] = train_df['label'].astype(int)
test_df['label'] = test_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])
val_dataset = Dataset.from_pandas(val_df[['text', 'label']])

def tokenize_function(examples):
    return filter_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Re-initialize the model to output 3 classes and support classification loss calculation
filter_model = AutoModelForSequenceClassification.from_pretrained(
    web_filter_model_name,
    num_labels=3,
    ignore_mismatched_sizes=True,
    device_map="cuda:0"
)
filter_model = filter_model.eval()

# 2. Configure training arguments
training_args = TrainingArguments(
    output_dir="./web_filter_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=4e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=False, # Changed to False to fix NotImplementedError
    report_to="none"
)

# 3. Initialize Trainer
# We use 'processing_class' instead of 'tokenizer' for this transformers version
trainer = Trainer(
    model=filter_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=filter_tokenizer,
)

# 4. Start training
print("Starting WebFilter training for 3 classes...")
trainer.train()

Map:   0%|          | 0/43785 [00:00<?, ? examples/s]

Map:   0%|          | 0/12562 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: dayll/WebFilter-7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting WebFilter training for 3 classes...


Epoch,Training Loss,Validation Loss
1,1.498680,0.739517


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21893, training_loss=1.0578592900561636, metrics={'train_runtime': 7872.2338, 'train_samples_per_second': 5.562, 'train_steps_per_second': 2.781, 'total_flos': 8.777466456204902e+17, 'train_loss': 1.0578592900561636, 'epoch': 1.0})

In [26]:
# 1. Define custom RAG pipeline using your newly trained WebFilter model
def custom_filter_documents(query, retrieved_docs, model, tokenizer, threshold=0.5):
    """
    Filters documents using a specified WebFilter model (original or newly trained).
    """
    filtered_docs = []
    for doc in retrieved_docs:
        inputs = tokenizer(
            f"Query: {query} Document: {doc}",
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            # Label mapping: 0 -> Negative/Noise, 1 -> Neutral, 2 -> Positive/Clean
            # For filtering, we want relevant/clean classes (e.g. neutral/positive score combined)
            relevance_score = (probs[0][1] + probs[0][2]).item()

        if relevance_score >= threshold:
            filtered_docs.append(doc)
    return filtered_docs

def fingpt_pipeline_inference(query, retrieved_docs=None, filter_model_obj=None, filter_tokenizer_obj=None, threshold=0.5):
    """
    Runs the FinGPT response generation.
    - If retrieved_docs is None, runs the base FinGPT model without any context.
    - If filter_model_obj is None, it bypasses filtering.
    """
    if retrieved_docs is None:
        # Standard baseline query without context - matching FinGPT training format exactly
        prompt = f"""Instruction: What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}
Input: {query}
Answer: """
    else:
        if filter_model_obj is not None and filter_tokenizer_obj is not None:
            clean_docs = custom_filter_documents(query, retrieved_docs, filter_model_obj, filter_tokenizer_obj, threshold)
        else:
            clean_docs = retrieved_docs

        context = " ".join(clean_docs) if clean_docs else "No additional context."
        # Inserting context inside the exact FinGPT expected Input format
        prompt = f"""Instruction: What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}
Input: Context: {context} | Query: {query}
Answer: """

    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = fingpt_model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the newly generated response text after our prompt
    if "Answer:" in response:
        ans = response.split("Answer:")[-1].strip().lower()
    else:
        ans = response.replace(prompt, "").strip().lower()

    # Fallback cleanup to extract just the classification keyword
    for clean_word in ["negative", "neutral", "positive"]:
        if clean_word in ans:
            return clean_word

    return ans

def live_web_search(query):
    """
    Perform a live web search using a highly reliable, unblockable Yahoo Finance RSS Feed Parser.
    Uses a smart keyword reduction strategy to guarantee news search results are always found.
    """
    import urllib.parse
    import requests
    import re
    from bs4 import BeautifulSoup

    def fetch_rss(search_term):
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36"}
        try:
            url = f"https://news.search.yahoo.com/rss?p={urllib.parse.quote(search_term)}"
            r = requests.get(url, headers=headers, timeout=5)
            if r.status_code == 200:
                descriptions = re.findall(r"<description>(.*?)</description>", r.text, re.DOTALL)
                out = []
                for desc in descriptions[1:4]:
                    clean_text = BeautifulSoup(desc, "html.parser").get_text().strip()
                    clean_text = re.sub(r"\s*-\s*[^-]+$", "", clean_text)
                    if clean_text:
                        out.append(clean_text)
                return out
        except Exception:
            pass
        return []

    # Clean the input query
    clean_query = re.sub(r"https?:\S+", "", query)
    clean_query = re.sub(r"\$\w+", "", clean_query)
    clean_query = re.sub(r"[^\w\s-]", " ", clean_query)
    clean_query = " ".join(clean_query.split()).strip()

    if not clean_query:
        return []

    # Extract distinct proper nouns (Capitalized words) to serve as prime targets
    keywords = re.findall(r"\b[A-Z][a-zA-Z0-9_]*\b", query)
    # Filter out common stop capitalized words if any exist
    stop_caps = {"Yahoo", "Finance", "CDP", "Instruction", "Input", "Answer", "Context", "Query"}
    keywords = [k for k in keywords if k not in stop_caps]

    # Choose the top 2 unique proper nouns, or fallback to the first 2 words of the cleaned text query
    if len(keywords) >= 1:
        target_keywords = keywords[:2]
    else:
        target_keywords = clean_query.split()[:2]

    search_term = " ".join(target_keywords).strip()
    results = []
    if search_term:
        results = fetch_rss(search_term)

    return results

# 2. Run batch testing on the entire testing dataset to compare accuracy rates
print(f"=== Starting Complete Test Set Evaluation (Total: {len(test_df)} samples) ===\n")

y_true = []
y_base = []
y_unfiltered = []
y_filtered = []

for idx, row in test_df.iterrows():
    query = row['text']
    true_sentiment = str(row['output']).strip().lower()

    # Retrieve real web documents dynamically via search engine scraper
    live_docs = live_web_search(query)
    if not live_docs:
        live_docs = ["No live web results available"]

    # A. Base FinGPT model without context (Zero-shot baseline)
    base_ans = fingpt_pipeline_inference(query, retrieved_docs=None)

    # B. Bypassing Web Filter entirely (Raw unfiltered live context)
    raw_ans = fingpt_pipeline_inference(query, live_docs, filter_model_obj=None)

    # C. Using the newly trained Web Filter model
    trained_ans = fingpt_pipeline_inference(
        query,
        live_docs,
        filter_model_obj=filter_model,
        filter_tokenizer_obj=filter_tokenizer
    )

    # Clean and append results
    base_pred_clean = base_ans.strip().lower()
    unfiltered_pred_clean = raw_ans.strip().lower()
    filtered_pred_clean = trained_ans.strip().lower()

    y_true.append(true_sentiment)
    y_base.append(base_pred_clean)
    y_unfiltered.append(unfiltered_pred_clean)
    y_filtered.append(filtered_pred_clean)

    if (idx + 1) % 50 == 0 or (idx + 1) == len(test_df):
        print(f"Processed {idx + 1}/{len(test_df)} samples...")

# Calculate accuracy percentages
accuracy_base = sum([1 for yt, yp in zip(y_true, y_base) if yt in yp]) / len(y_true) * 100
accuracy_unfiltered = sum([1 for yt, yp in zip(y_true, y_unfiltered) if yt in yp]) / len(y_true) * 100
accuracy_filtered = sum([1 for yt, yp in zip(y_true, y_filtered) if yt in yp]) / len(y_true) * 100

# Display overall testing metrics
print("\n=================== TEST SET ACCURACY METRICS ===================")
print(f"Base FinGPT (No Context) Accuracy: {accuracy_base:.2f}%")
print(f"FinGPT + Raw Live Context Accuracy: {accuracy_unfiltered:.2f}%")
print(f"FinGPT + Filtered Context (Trained WebFilter) Accuracy: {accuracy_filtered:.2f}%")
print("==================================================================\n")

=== Starting Pipeline Evaluation and Live Comparisons ===

Test Sample #1
Query: Kesko has about 2,000 stores engaged in chain operations in the Nordic and Baltic countries , Russia , and Belarus .
Ground Truth Sentiment: neutral
Live Retrieved Docs: ['No live web results available for search query: Kesko has about 2,000 stores engaged in chain operations in the Nordic and Baltic countries , Russia , and Belarus .']
[*] Base FinGPT (No Context): neutral
[-] FinGPT with Raw Unfiltered Context: neutral
[+] FinGPT with Trained WebFilter Context: neutral
--------------------------------------------------------------------------------
Test Sample #2
Query: CDP was established on the initiative of institutional investors ; however , the annually published results also interest an increasing number of customers and other interest groups of the reporting companies .
Ground Truth Sentiment: neutral
Live Retrieved Docs: ['No live web results available for search query: CDP was established on the

In [29]:
# Results on Validation Set

# Ensure unique index alignment to prevent duplicate counts during iteration
eval_df = val_df.reset_index(drop=True)

# Prepare to gather results for comparative dataframe
comparison_results = []

print(f"=== Running Batch Comparison of Model Pipelines on Validation Set (Total: {len(eval_df)} samples) ===\n")

# Initialize lists to track predictions and ground truths for accuracy
y_true = []
y_base = []
y_unfiltered = []
y_filtered = []

for idx, row in eval_df.iterrows():
    query = row['text']
    true_sentiment = str(row['output']).strip().lower()

    # 1. Fetch live web snippets dynamically
    live_docs = live_web_search(query)
    if not live_docs:
        live_docs = ["No live web results available"]

    # 2. Get predictions from all three configurations
    base_pred = fingpt_pipeline_inference(query, retrieved_docs=None)
    unfiltered_pred = fingpt_pipeline_inference(query, live_docs, filter_model_obj=None)
    filtered_pred = fingpt_pipeline_inference(
        query,
        live_docs,
        filter_model_obj=filter_model,
        filter_tokenizer_obj=filter_tokenizer
    )

    # Normalize outputs for metrics comparison
    base_pred_clean = base_pred.strip().lower()
    unfiltered_pred_clean = unfiltered_pred.strip().lower()
    filtered_pred_clean = filtered_pred.strip().lower()

    y_true.append(true_sentiment)
    y_base.append(base_pred_clean)
    y_unfiltered.append(unfiltered_pred_clean)
    y_filtered.append(filtered_pred_clean)

    # 3. Append comparative statistics and outputs
    comparison_results.append({
        "Query": query[:100] + "..." if len(query) > 100 else query,
        "Ground Truth": true_sentiment,
        "Base FinGPT (No Context)": base_pred,
        "FinGPT + Raw Live Context": unfiltered_pred,
        "FinGPT + Filtered Context (Trained WebFilter)": filtered_pred
    })

    if (idx + 1) % 50 == 0 or (idx + 1) == len(eval_df):
        print(f"Processed {idx + 1}/{len(eval_df)} samples...")

# Create a structured comparative DataFrame
comparison_df = pd.DataFrame(comparison_results)

# Calculate accuracy percentages
accuracy_base = sum([1 for yt, yp in zip(y_true, y_base) if yt in yp]) / len(y_true) * 100
accuracy_unfiltered = sum([1 for yt, yp in zip(y_true, y_unfiltered) if yt in yp]) / len(y_true) * 100
accuracy_filtered = sum([1 for yt, yp in zip(y_true, y_filtered) if yt in yp]) / len(y_true) * 100

# Display overall metrics
print("\n=================== ACCURACY METRICS ===================")
print(f"Base FinGPT (No Context) Accuracy: {accuracy_base:.2f}%")
print(f"FinGPT + Raw Live Context Accuracy: {accuracy_unfiltered:.2f}%")
print(f"FinGPT + Filtered Context (Trained WebFilter) Accuracy: {accuracy_filtered:.2f}%")
print("========================================================\n")

# Display the structured results table
display(comparison_df.head(20))

=== Running Batch Comparison of Model Pipelines on Validation Set (Total: 6205 samples) ===

Processed 50/6205 samples...
Processed 100/6205 samples...
Processed 150/6205 samples...
Processed 200/6205 samples...
Processed 250/6205 samples...
Processed 300/6205 samples...
Processed 350/6205 samples...
Processed 400/6205 samples...
Processed 450/6205 samples...
Processed 500/6205 samples...
Processed 550/6205 samples...
Processed 600/6205 samples...
Processed 650/6205 samples...
Processed 700/6205 samples...
Processed 750/6205 samples...
Processed 800/6205 samples...
Processed 850/6205 samples...
Processed 900/6205 samples...
Processed 950/6205 samples...
Processed 1000/6205 samples...
Processed 1050/6205 samples...
Processed 1100/6205 samples...
Processed 1150/6205 samples...
Processed 1200/6205 samples...
Processed 1250/6205 samples...
Processed 1300/6205 samples...
Processed 1350/6205 samples...
Processed 1400/6205 samples...
Processed 1450/6205 samples...
Processed 1500/6205 samples.

,Query,Ground Truth,Base FinGPT (No Context),FinGPT + Raw Live Context,FinGPT + Filtered Context (Trained WebFilter)
0,Berkshire applies to boost Wells Fargo stake a...,positive,positive,positive,positive
1,"From what you buy online, to how you remember ...",neutral,neutral,neutral,neutral
2,ouch looks like $tsla shorts getting skinned a...,positive,positive,positive,positive
3,"Red Daily Triangle on $SHOR,...Scaling Up on L...",positive,positive,positive,positive
4,GlaxoSmithKline share price slips as FDA okays...,negative,negative,negative,negative
5,Canada racks up 400-tonne cannabis pile after ...,neutral,negative,neutral,neutral
6,The solution is demonstrated on a tablet devel...,neutral,neutral,neutral,neutral
7,The plant is scheduled for completion in late ...,neutral,neutral,neutral,neutral
8,$EBAY - EBay +6% after selling StubHub https:/...,positive,positive,positive,positive
9,The decision of the French Court relates to th...,neutral,neutral,neutral,neutral
